# CS781 Phase 1 v3 — four different foundation-model architectures on one shared tokenizer, corpus and recipe

The spec asks for **four different architectures** (no specific models named) and allows a `[MASK]` token. This notebook pre-trains one model from each family, all inside the 50M-300M parameter range:

| # | family | model (Hugging Face / PyTorch) | objective | params | why |
|---|---|---|---|---|---|
| 1 | **decoder-only** Transformer (causal) | `GPT2LMHeadModel`, 12 x 768 | next-token prediction | 110.0M | the standard LLM design; your existing model's family |
| 2 | **encoder-only** Transformer (bidirectional) | `ModernBertForMaskedLM`, 12 x 768, RoPE, GeGLU, local+global attention | masked LM, 30% masking | 85.4M | encoders beat decoders at classification and tagging when trained with the same recipe (Ettin, BabyLM, Cramming) - your Phase 2/3 tasks |
| 3 | **encoder-decoder** Transformer | `T5ForConditionalGeneration`, 6+6 x 640, gated-GELU, relative positions | span denoising (one `[MASK]` sentinel) | 91.3M | a different information flow; its encoder alone works for classification (EncT5) |
| 4 | **recurrent** (non-Transformer) | bidirectional LSTM LM (ELMo-style), 3 layers x 1024 | forward + backward next-token | 63.7M | no attention at all: a genuinely different inductive bias, installs nothing, and has a track record on tagging (ELMo) |

Mamba was considered for #4 but its fast path needs compiled CUDA kernels that often fail to install on Kaggle; the LSTM is the low-risk non-attention choice.

## What was checked before you run anything
* On CPU with tiny versions of all four: forward/backward, **T5 corruption reconstructs the text exactly (200/200)**, MLM statistics (30% selected, 80/10/10), the BiLSTM's forward direction cannot see the future, its backward direction cannot see the past, and padding does not change real positions; each model then learns a predictable pattern; the trainer resumes; the loader module loads all four checkpoints.
* One real bug found and fixed: with the usual tiny embedding init the BiLSTM stalled at the unigram entropy (learned *which* tokens occur, not what follows what). The embedding init and LayerNorm are changed accordingly.
* Real-size configs were built with the exact calls used here and their parameter counts verified. **Not tested: GPU speed and memory, and fp16 stability on a T4** - cell 6 (smoke test) is the first real test.

## Shared recipe
* One tokenizer: your 32,000-token legal BPE plus **one** new special token `[MASK]` (id 32000, vocabulary 32,001). It is saved as `legal_bpe_32k_mask.json`; attach that file to the Phase 2/3 notebooks.
* Corpus from `cs781-phase4-training-prep`, validation/test documents never trained on, domain-weighted (30% Supreme Court, 30% tribunals, 35% high courts, 5% acts), shuffled, packed into 512-token blocks with no padding, 131,072 tokens per optimizer step, AdamW (beta2 0.95, weight decay 0.1 on matrices), warmup-stable-decay schedule, automatic resume.
* Validation loss per domain on fixed held-out blocks (masking/corruption use a fixed seed). **Losses are not comparable across architectures** - compare them by downstream F1 in the Phase 2/3 notebooks.

## Run order
1 setup -> 2 data -> 3 models -> 4 objectives -> 5 trainer -> **6 smoke test (must pass)** -> 7a encoder -> 7b decoder -> 7c encoder-decoder -> 7d recurrent -> 8 export.
Suggested GPU-hours: encoder 5, decoder 3, encoder-decoder 3, recurrent 2 (change `HOURS` in cell 1). Token counts are printed by each run once it has measured your real speed.

## After training
Cell 8 writes `arch_loaders.py`: `load_backbone(path)` returns the same interface for all four families. The Phase 3 notebook needs a small update to use it (ModernBERT's q/k/v are one fused `Wqkv` layer, T5's are separate `q`/`k`/`v`, the LSTM supports full fine-tuning only) - ask me for it once you have checkpoints.


In [ ]:
# ================= CELL 1 — setup, paths, tokenizer (+[MASK]), per-architecture configuration =================
import os, sys, re, json, math, time, random, glob, gzip, gc, copy
from pathlib import Path
from collections import defaultdict, Counter
import numpy as np, pandas as pd, torch, torch.nn as nn

BASE    = "/kaggle/input/datasets/oipranesh/cs781-phase4-training-prep"          # dedup corpus, manifests, tokenizer
P1_CKPT = "/kaggle/input/datasets/oipranesh/cs781-phase-1-checkpoints/checkpoint_latest.pt"    # only needed for gpt2 with init='warm3000'
OUT = Path("/kaggle/working/pretrain_4arch"); OUT.mkdir(parents=True, exist_ok=True)
assert os.path.exists(BASE), f"missing: {BASE}"
REQUIRE_GPU, TINY = True, False
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if REQUIRE_GPU: assert device.type == "cuda", "pretraining needs a GPU"
USE_BF16 = device.type == "cuda" and torch.cuda.get_device_capability(0)[0] >= 8      # T4 (7.5) has no native bf16 -> fp16 + GradScaler
AMP = device.type == "cuda"
print("device:", torch.cuda.get_device_name(0) if device.type == "cuda" else "cpu", "| precision:", ("bf16" if USE_BF16 else "fp16 + GradScaler") if AMP else "fp32")

# ---- tokenizer: the original 32,000-token legal BPE plus ONE new special token, [MASK] (id 32000) ----
from tokenizers import Tokenizer
tok = Tokenizer.from_file(f"{BASE}/tokenizer/legal_bpe_32k.json"); n0 = tok.get_vocab_size()
PAD, UNK, BOS, EOS = (tok.token_to_id(t) for t in ("[PAD]", "[UNK]", "[BOS]", "[EOS]")); assert None not in (PAD, UNK, BOS, EOS)
if tok.token_to_id("[MASK]") is None: tok.add_special_tokens(["[MASK]"])
MASK = tok.token_to_id("[MASK]"); V = tok.get_vocab_size(); assert MASK == n0 and V == n0 + 1, (MASK, V, n0)
tok.save(str(OUT / "legal_bpe_32k_mask.json"))
print(f"tokenizer: {n0} -> {V} tokens | PAD/UNK/BOS/EOS/MASK = {PAD}/{UNK}/{BOS}/{EOS}/{MASK} | saved {OUT/'legal_bpe_32k_mask.json'} (attach this file to the Phase 2/3 notebooks)")

DOMAINS = ["supreme_court", "tribunals", "high_courts", "acts"]
SEQ = 512                               # pre-training block length for every architecture (downstream windows are <= 512 tokens)
COMMON = dict(lr_min_frac=0.1, warmup=100, wd=0.1, betas=(0.9, 0.95), clip=1.0, eval_every=200, eval_tokens_per_domain=262144, min_words=50, seed=1234,
              domain_w={"supreme_court": 0.30, "tribunals": 0.30, "high_courts": 0.35, "acts": 0.05})     # downstream texts are judgments, not statutes
ARCH_CFG = {    # tokens per optimizer step = micro_bs * SEQ * accum = 131,072 for every architecture
    "gpt2":       dict(micro_bs=16, accum=16, lr_peak=6e-4, amp=True, init="scratch"),    # init='warm3000' continues the old 3,000-step weights
    "modernbert": dict(micro_bs=16, accum=16, lr_peak=6e-4, amp=True),
    "t5":         dict(micro_bs=16, accum=16, lr_peak=8e-4, amp=True),
    "bilstm":     dict(micro_bs=8,  accum=32, lr_peak=1e-3, amp=True)}
HOURS = {"modernbert": 5.0, "gpt2": 3.0, "t5": 3.0, "bilstm": 2.0}      # suggested GPU-hour split (about 13 h); the encoder is the most useful, so it gets the most


In [ ]:
# ================= CELL 2 — streaming corpus: held-out split, domain-weighted shuffled mix, packing WITHOUT padding =================
def shard_files(domain):
    fs = sorted(glob.glob(f"{BASE}/dedup/{domain}/*.jsonl") + glob.glob(f"{BASE}/dedup/{domain}/*.jsonl.gz")); assert fs, f"no shards for {domain}"; return fs
def open_text(p): return gzip.open(p, "rt", encoding="utf-8") if p.endswith(".gz") else open(p, encoding="utf-8")
def load_hash_set(split):
    s = set()
    with open(f"{BASE}/manifests/{split}_manifest.jsonl", encoding="utf-8") as f:
        for line in f: s.add(json.loads(line)["text_hash"])
    return s
HELD = {sp: load_hash_set(sp) for sp in ("validation", "test")}
SKIP = HELD["validation"] | HELD["test"]               # training never sees validation or test documents
print({k: f"{len(v):,} docs" for k, v in HELD.items()}, "| shards:", {d: len(shard_files(d)) for d in DOMAINS})

def iter_docs(domain, rng, skip, min_words):
    files = list(shard_files(domain))
    while True:
        rng.shuffle(files)
        for p in files:
            with open_text(p) as f:
                for line in f:
                    try: r = json.loads(line)
                    except Exception: continue
                    if r.get("text_hash") in skip: continue
                    t = r.get("text")
                    if not t: continue
                    try: wc = int(r.get("word_count"))
                    except Exception: wc = len(t.split())
                    if wc >= min_words: yield t

def shuffled(it, buf, rng):                              # shard files are ordered, so mix them through a buffer
    pool = []
    for x in it:
        pool.append(x)
        if len(pool) >= buf:
            i = rng.randrange(len(pool)); pool[i], pool[-1] = pool[-1], pool[i]; yield pool.pop()

class PackedStream:
    """domain-weighted documents -> [BOS] tokens [EOS] -> packed into [micro_bs, SEQ] blocks with NO padding"""
    def __init__(self, tokenizer, micro_bs, seq, domain_w, min_words, seed, skip):
        self.tok, self.B, self.T, self.rng = tokenizer, micro_bs, seq, random.Random(seed); self.buf = []; self.dom_tokens = Counter()
        self.gens = {d: shuffled(iter_docs(d, random.Random(seed + i), skip, min_words), 1000, random.Random(seed + 100 + i)) for i, d in enumerate(DOMAINS)}
        self.w = [domain_w[d] for d in DOMAINS]
    def _refill(self, need):
        while len(self.buf) < need:
            ds = [self.rng.choices(DOMAINS, weights=self.w)[0] for _ in range(32)]; texts = [next(self.gens[d]) for d in ds]
            for d, enc in zip(ds, self.tok.encode_batch(texts, add_special_tokens=False)):
                ids = [BOS] + enc.ids + [EOS]; self.buf.extend(ids); self.dom_tokens[d] += len(ids)
    def next_batch(self):
        self._refill(self.B * self.T); x = torch.tensor(self.buf[:self.B * self.T], dtype=torch.long).view(self.B, self.T); del self.buf[:self.B * self.T]; return x

def build_val_blocks(per_domain_tokens, cache):
    """fixed held-out blocks per domain, from validation-manifest documents only, spread over several shards"""
    if cache.exists(): return torch.load(cache)
    per = per_domain_tokens // SEQ; need = per * SEQ; val = {}
    for d in DOMAINS:
        files = shard_files(d); idx = sorted({int(i * (len(files) - 1) / max(1, min(6, len(files)) - 1)) for i in range(min(6, len(files)))})
        quota = math.ceil(need / len(idx)); toks = []
        for fi in idx:
            got = 0
            with open_text(files[fi]) as f:
                for line in f:
                    try: r = json.loads(line)
                    except Exception: continue
                    if r.get("text_hash") in HELD["validation"] and r.get("text"):
                        ids = [BOS] + tok.encode(r["text"], add_special_tokens=False).ids + [EOS]; toks += ids; got += len(ids)
                        if got >= quota: break
        assert len(toks) >= need, f"{d}: only {len(toks)} validation tokens found (need {need})"
        val[d] = torch.tensor(toks[:need], dtype=torch.long).view(per, SEQ)
    torch.save(val, cache); return val
VAL = build_val_blocks(COMMON["eval_tokens_per_domain"], OUT / f"val_blocks_{SEQ}.pt")
print("validation blocks:", {d: tuple(v.shape) for d, v in VAL.items()}, "| tokens:", sum(v.numel() for v in VAL.values()))


In [ ]:
# ================= CELL 3 — the FOUR architectures (same tokenizer, same data, different inductive biases) =================
import torch.nn.functional as F
from transformers import GPT2Config, GPT2LMHeadModel, ModernBertConfig, ModernBertForMaskedLM, T5Config, T5ForConditionalGeneration

def specs(tiny=False):
    """hyper-parameters of every architecture (tiny=True is only used by the unit tests)"""
    if tiny:
        return dict(gpt2=dict(n_layer=2, n_embd=64, n_head=4, n_inner=256),
                    modernbert=dict(hidden_size=64, intermediate_size=96, num_hidden_layers=4, num_attention_heads=4, local_attention=16, global_attn_every_n_layers=2),
                    t5=dict(d_model=64, d_kv=16, d_ff=128, num_layers=2, num_decoder_layers=2, num_heads=4),
                    bilstm=dict(emb=32, hidden=48, layers=2))
    return dict(gpt2=dict(n_layer=12, n_embd=768, n_head=12, n_inner=3072),                                      # causal decoder-only Transformer
                modernbert=dict(hidden_size=768, intermediate_size=1152, num_hidden_layers=12, num_attention_heads=12, local_attention=128, global_attn_every_n_layers=3),   # bidirectional encoder-only Transformer
                t5=dict(d_model=640, d_kv=64, d_ff=1792, num_layers=6, num_decoder_layers=6, num_heads=10),   # encoder-decoder Transformer
                bilstm=dict(emb=512, hidden=1024, layers=3))                                                   # recurrent bidirectional LSTM language model (ELMo-style)
ARCHS = ["gpt2", "modernbert", "t5", "bilstm"]

class BiLSTMLM(nn.Module):
    """ELMo-style bidirectional LM: a forward LSTM predicts the next token, a backward LSTM predicts the previous token; embeddings are tied to the softmax.
       Embeddings start at N(0,1) and the LSTM outputs are LayerNorm-ed before the tied softmax: with the usual tiny init (std 0.02) this model sat on a long
       plateau at the unigram entropy in testing (it learned WHICH tokens occur, not what follows what)."""
    def __init__(self, vocab, emb=512, hidden=1024, layers=3):
        super().__init__()
        self.vocab, self.emb_dim, self.hidden, self.layers = vocab, emb, hidden, layers
        self.emb = nn.Embedding(vocab, emb)                      # default init N(0, 1)
        self.fwd = nn.LSTM(emb, hidden, layers, batch_first=True); self.bwd = nn.LSTM(emb, hidden, layers, batch_first=True)
        self.norm_f, self.norm_b = nn.LayerNorm(hidden), nn.LayerNorm(hidden)
        self.proj_f, self.proj_b = nn.Linear(hidden, emb), nn.Linear(hidden, emb); self.out_bias = nn.Parameter(torch.zeros(vocab))
    def _lm_loss(self, h, targets):
        logits = h @ self.emb.weight.t() + self.out_bias
        return F.cross_entropy(logits.float().reshape(-1, self.vocab), targets.reshape(-1))
    def forward(self, input_ids, labels=None):                 # pretraining: packed blocks, no padding
        x = input_ids; xb = torch.flip(x, [1])
        hf, _ = self.fwd(self.emb(x)); hb, _ = self.bwd(self.emb(xb))
        return 0.5 * (self._lm_loss(self.proj_f(self.norm_f(hf[:, :-1])), x[:, 1:]) + self._lm_loss(self.proj_b(self.norm_b(hb[:, :-1])), xb[:, 1:]))
    def hidden_states(self, input_ids, attention_mask=None):   # downstream: [B,T,2H]; right-padding safe (each row is reversed inside its own length)
        x = input_ids; B, T = x.shape
        lengths = attention_mask.sum(1) if attention_mask is not None else torch.full((B,), T, device=x.device)
        rev = (lengths[:, None] - 1 - torch.arange(T, device=x.device)[None, :]).clamp(min=0)
        hf, _ = self.fwd(self.emb(x)); hb, _ = self.bwd(self.emb(x.gather(1, rev)))
        hb = hb.gather(1, rev[:, :, None].expand(-1, -1, hb.shape[-1]))
        return torch.cat([hf, hb], dim=-1)

def build_model(arch, tiny=False, init="scratch"):
    s = specs(tiny)[arch]
    if arch == "gpt2":
        cfg = GPT2Config(vocab_size=V, n_positions=SEQ, n_ctx=SEQ, resid_pdrop=0.0, embd_pdrop=0.0, attn_pdrop=0.0, bos_token_id=BOS, eos_token_id=EOS, pad_token_id=PAD, use_cache=False, **s)
        m = GPT2LMHeadModel._from_config(cfg, attn_implementation="sdpa") if not tiny else GPT2LMHeadModel(cfg)
        if init == "warm3000":                                   # continue the old 3,000-step weights: copy 32,000 embedding rows, add a [MASK] row, keep the first SEQ positions
            ck = torch.load(P1_CKPT, map_location="cpu", weights_only=False); sd = ck["model_state_dict"]; wte = sd["transformer.wte.weight"]
            wte = torch.cat([wte, wte.mean(0, keepdim=True)], 0); sd["transformer.wte.weight"] = wte; sd["lm_head.weight"] = wte
            sd["transformer.wpe.weight"] = sd["transformer.wpe.weight"][:SEQ].clone(); print("warm start:", m.load_state_dict(sd, strict=True)); del ck, sd
        mc = dict(family="gpt2", vocab_size=V, context_length=SEQ, hidden_size=s["n_embd"], num_layers=s["n_layer"], num_heads=s["n_head"], n_inner=s["n_inner"])
    elif arch == "modernbert":
        cfg = ModernBertConfig(vocab_size=V, max_position_embeddings=SEQ, pad_token_id=PAD, bos_token_id=BOS, eos_token_id=EOS, cls_token_id=BOS, sep_token_id=EOS,
                               attention_dropout=0.0, mlp_dropout=0.0, embedding_dropout=0.0, classifier_pooling="mean", sparse_prediction=True, sparse_pred_ignore_index=-100, reference_compile=False, **s)
        m = ModernBertForMaskedLM._from_config(cfg, attn_implementation="sdpa") if not tiny else ModernBertForMaskedLM(cfg)
        mc = dict(family="modernbert", vocab_size=V, context_length=SEQ, hidden_size=s["hidden_size"], intermediate_size=s["intermediate_size"], num_layers=s["num_hidden_layers"],
                  num_heads=s["num_attention_heads"], local_attention=s["local_attention"], global_attn_every_n_layers=s["global_attn_every_n_layers"])
    elif arch == "t5":
        cfg = T5Config(vocab_size=V, feed_forward_proj="gated-gelu", dropout_rate=0.0, tie_word_embeddings=True, pad_token_id=PAD, eos_token_id=EOS, decoder_start_token_id=PAD, **s)
        m = T5ForConditionalGeneration(cfg)
        mc = dict(family="t5", vocab_size=V, context_length=SEQ, hidden_size=s["d_model"], d_kv=s["d_kv"], d_ff=s["d_ff"], num_layers=s["num_layers"], num_decoder_layers=s["num_decoder_layers"], num_heads=s["num_heads"])
    elif arch == "bilstm":
        m = BiLSTMLM(V, **s); mc = dict(family="bilstm", vocab_size=V, context_length=SEQ, hidden_size=2 * s["hidden"], emb=s["emb"], lstm_hidden=s["hidden"], num_layers=s["layers"])
    else: raise ValueError(arch)
    if hasattr(m, "config"): m.config.use_cache = False
    n = sum(p.numel() for p in m.parameters())
    if not tiny: assert 50e6 <= n <= 300e6, f"{arch}: {n:,} parameters is outside the 50M-300M Phase 1 range"
    return (m if tiny else m.to(device)), mc, n


In [ ]:
# ================= CELL 4 — training objectives: CLM (GPT-2), MLM (encoder), span denoising (encoder-decoder), bidirectional LM (BiLSTM) =================
SPECIAL = [PAD, UNK, BOS, EOS]

def mlm_mask(x, p=0.30, gen=None):
    """BERT-style: select p of the non-special tokens; 80% -> [MASK], 10% -> random token, 10% unchanged. labels = -100 elsewhere."""
    sel = torch.rand(x.shape, generator=gen, device=x.device) < p
    for s in SPECIAL: sel &= (x != s)
    labels = torch.where(sel, x, torch.full_like(x, -100))
    r = torch.rand(x.shape, generator=gen, device=x.device); inp = x.clone()
    inp[sel & (r < 0.8)] = MASK
    rnd = sel & (r >= 0.8) & (r < 0.9); inp[rnd] = torch.randint(4, MASK, x.shape, generator=gen, device=x.device)[rnd]
    return inp, labels

def random_spans_noise_mask(length, noise_density, mean_span, rng):
    """T5's span sampler (numpy): returns a boolean array marking the noise tokens"""
    n_noise = min(max(int(np.round(length * noise_density)), 1), length - 1); n_spans = max(int(np.round(n_noise / mean_span)), 1)
    def seg(items, segments):
        m = np.arange(items - 1) < (segments - 1); rng.shuffle(m); first = np.pad(m, [[1, 0]]); ids = np.cumsum(first)
        return np.unique(ids, return_counts=True)[1]
    noise_len, keep_len = seg(n_noise, n_spans), seg(length - n_noise, n_spans)
    inter = np.reshape(np.stack([keep_len, noise_len], axis=1), [n_spans * 2]); starts = np.cumsum(inter)[:-1]
    ind = np.zeros(length, dtype=bool); ind[starts] = True
    return (np.cumsum(ind) % 2 == 1)[:length]

def t5_corrupt_one(tokens, rng, noise_density=0.15, mean_span=3.0):
    """each noise span -> ONE [MASK] in the input; target = [MASK] span [MASK] span ... [EOS] (a single sentinel type, so no extra vocabulary is needed)"""
    noise = random_spans_noise_mask(len(tokens), noise_density, mean_span, rng); inp, tgt, i, L = [], [], 0, len(tokens)
    while i < L:
        if noise[i]:
            j = i
            while j < L and noise[j]: j += 1
            inp.append(MASK); tgt.append(MASK); tgt.extend(tokens[i:j]); i = j
        else: inp.append(tokens[i]); i += 1
    tgt.append(EOS); return inp, tgt

def t5_batch(x, rng):
    xs = x.cpu().numpy(); pairs = [t5_corrupt_one(r.tolist(), rng) for r in xs]
    Li, Lt = max(len(a) for a, _ in pairs), max(len(b) for _, b in pairs)
    inp = torch.full((len(pairs), Li), PAD, dtype=torch.long); att = torch.zeros_like(inp); lab = torch.full((len(pairs), Lt), -100, dtype=torch.long)
    for k, (a, b) in enumerate(pairs): inp[k, :len(a)] = torch.tensor(a); att[k, :len(a)] = 1; lab[k, :len(b)] = torch.tensor(b)
    return inp.to(x.device), att.to(x.device), lab.to(x.device)

_np_rng = np.random.default_rng(0)
def forward_loss(arch, model, x, rng=None, gen=None):
    """x: [B, SEQ] packed token blocks -> scalar loss for the architecture's own objective"""
    if arch == "gpt2": return model(input_ids=x, labels=x).loss
    if arch == "modernbert":
        inp, labels = mlm_mask(x, 0.30, gen); return model(input_ids=inp, attention_mask=torch.ones_like(inp), labels=labels).loss
    if arch == "t5":
        inp, att, lab = t5_batch(x, rng or _np_rng); return model(input_ids=inp, attention_mask=att, labels=lab).loss
    if arch == "bilstm": return model(x)
    raise ValueError(arch)


In [ ]:
# ================= CELL 5 — evaluation + trainer (warmup-stable-decay, fp16/bf16, resume, per-domain validation) =================
import contextlib
def amp_ctx(enabled=True):
    if not (AMP and enabled): return contextlib.nullcontext()
    return torch.autocast("cuda", dtype=torch.bfloat16 if USE_BF16 else torch.float16)

_VALB = {}
def val_batches(arch, bs=16, max_batches=None):
    """fixed, deterministic validation inputs per architecture (masking / span corruption use a fixed seed, so every evaluation is comparable)"""
    key = (arch, bs, max_batches)
    if key not in _VALB:
        gen = torch.Generator().manual_seed(123); rng = np.random.default_rng(123); out = {}
        for d, blocks in VAL.items():
            items = []
            for i in range(0, len(blocks), bs):
                if max_batches and len(items) >= max_batches: break
                x = blocks[i:i + bs]
                items.append(mlm_mask(x, 0.30, gen) if arch == "modernbert" else t5_batch(x, rng) if arch == "t5" else (x,))
            out[d] = items
        _VALB[key] = out
    return _VALB[key]

@torch.no_grad()
def evaluate(arch, model, amp=True, bs=16, max_batches=None):
    model.eval(); res = {}
    for d, items in val_batches(arch, bs, max_batches).items():
        tot = 0.0
        for it in items:
            it = [t.to(device) for t in it]
            with amp_ctx(amp):
                if arch == "gpt2": loss = model(input_ids=it[0], labels=it[0]).loss
                elif arch == "modernbert": loss = model(input_ids=it[0], attention_mask=torch.ones_like(it[0]), labels=it[1]).loss
                elif arch == "t5": loss = model(input_ids=it[0], attention_mask=it[1], labels=it[2]).loss
                else: loss = model(it[0])
            tot += float(loss)
        res[d] = tot / max(1, len(items))
    res["overall"] = float(np.mean([res[d] for d in VAL])); model.train(); return res
def fmt(res): return " | ".join(f"{k} {v:.3f}" for k, v in res.items())

def lr_at(step, c):
    w, T = c["warmup"], c["total_steps"]; ds = c["decay_start"] or int(0.7 * T); lo = c["lr_peak"] * c["lr_min_frac"]
    if step < w: return c["lr_peak"] * (step + 1) / w
    if step < ds: return c["lr_peak"]
    return c["lr_peak"] + (lo - c["lr_peak"]) * min(1.0, (step - ds) / max(1, T - ds))

def train(arch, cfg=None, steps=None, accum=None, do_eval=True, save=True, resume=True, minutes_budget=None, max_val_batches=None, keep_model=False, quiet=False, run=None):
    c = {**COMMON, **ARCH_CFG[arch], **(cfg or {})}; c.setdefault("total_steps", 3000); c.setdefault("decay_start", None); c["accum"] = accum or c["accum"]
    rdir = OUT / (run or arch); rdir.mkdir(exist_ok=True); torch.manual_seed(c["seed"])
    model, mc, n_params = build_model(arch, tiny=TINY, init=c.get("init", "scratch")); model.train()
    groups = [{"params": [p for p in model.parameters() if p.ndim >= 2], "weight_decay": c["wd"]}, {"params": [p for p in model.parameters() if p.ndim < 2], "weight_decay": 0.0}]
    try: opt = torch.optim.AdamW(groups, lr=c["lr_peak"], betas=c["betas"], eps=1e-8, fused=(device.type == "cuda"))
    except Exception: opt = torch.optim.AdamW(groups, lr=c["lr_peak"], betas=c["betas"], eps=1e-8)
    use_amp = AMP and c["amp"]; scaler = torch.amp.GradScaler("cuda", enabled=use_amp and not USE_BF16); step, tokens = 0, 0; ck_path = rdir / "latest.pt"
    if save and resume and ck_path.exists():
        ck = torch.load(ck_path, map_location="cpu", weights_only=False); model.load_state_dict(ck["model_state_dict"]); opt.load_state_dict(ck["optimizer_state_dict"])
        scaler.load_state_dict(ck["scaler_state_dict"]); step, tokens = ck["step"], ck["tokens"]; print(f"[{arch}] resumed from step {step}"); del ck
    stream = PackedStream(tok, c["micro_bs"], SEQ, c["domain_w"], c["min_words"], c["seed"] + step, SKIP)
    rng = np.random.default_rng(c["seed"] + step); gen = torch.Generator(device=device).manual_seed(c["seed"] + step)
    T = c["micro_bs"] * SEQ * c["accum"]; end = steps if steps is not None else c["total_steps"]; log = open(rdir / "log.jsonl", "a"); t0 = time.time(); last_save = time.time()
    best, run_loss, nb, step_times, hist, bad = float("inf"), 0.0, 0, [], [], 0; ev = c["eval_every"]
    while step < end:
        ts = time.time(); lr = lr_at(step, c)
        for g in opt.param_groups: g["lr"] = lr
        good = 0
        for _ in range(c["accum"]):
            x = stream.next_batch().to(device)
            with amp_ctx(use_amp): loss = forward_loss(arch, model, x, rng=rng, gen=gen)
            if not torch.isfinite(loss): bad += 1; print(f"[{arch}] non-finite loss (total {bad}); skipping microbatch"); assert bad < 50, "too many non-finite losses: try amp=False for this architecture"; continue
            scaler.scale(loss / c["accum"]).backward(); run_loss += loss.item(); nb += 1; good += 1
        if good:
            scaler.unscale_(opt); gn = torch.nn.utils.clip_grad_norm_(model.parameters(), c["clip"]); scaler.step(opt); scaler.update()
        else: gn = torch.tensor(float("nan"))
        opt.zero_grad(set_to_none=True); step += 1; tokens += T
        if device.type == "cuda": torch.cuda.synchronize()
        step_times.append(time.time() - ts)
        if step % 10 == 0:
            rec = dict(step=step, loss=run_loss / max(nb, 1), lr=lr, grad_norm=float(gn), tokens=tokens, tok_s=T / float(np.median(step_times[-10:])), elapsed_min=(time.time() - t0) / 60); run_loss, nb = 0.0, 0
            log.write(json.dumps(rec) + "\n"); log.flush()
            if not quiet and (step % 50 == 0 or step <= 20): print(f"[{arch}] step {step:5d} | loss {rec['loss']:.3f} | lr {lr:.2e} | gnorm {rec['grad_norm']:.2f} | {rec['tok_s']/1e3:.1f}k tok/s | {rec['elapsed_min']:.1f} min | {tokens/1e6:.0f}M tokens", flush=True)
        if do_eval and (step % ev == 0 or step == end):
            r = evaluate(arch, model, c["amp"], max_batches=max_val_batches); hist.append(dict(step=step, tokens=tokens, elapsed_min=(time.time() - t0) / 60, val=r)); log.write(json.dumps(dict(step=step, val=r)) + "\n"); log.flush()
            if not quiet: print(f"   [{arch}] VALIDATION step {step}: {fmt(r)}", flush=True)
            if save and r["overall"] < best: best = r["overall"]; torch.save({"model_state_dict": model.state_dict(), "model_config": mc, "step": step, "val": r}, rdir / "best.pt")
        if save and time.time() - last_save > 20 * 60:
            torch.save({"model_state_dict": model.state_dict(), "optimizer_state_dict": opt.state_dict(), "scaler_state_dict": scaler.state_dict(), "step": step, "tokens": tokens, "model_config": mc, "cfg": c}, ck_path)
            last_save = time.time(); print(f"   [{arch}] checkpoint saved")
        if minutes_budget and (time.time() - t0) / 60 > minutes_budget: print(f"[{arch}] time budget reached at step {step}"); break
    if save:
        torch.save({"model_state_dict": model.state_dict(), "optimizer_state_dict": opt.state_dict(), "scaler_state_dict": scaler.state_dict(), "step": step, "tokens": tokens, "model_config": mc, "cfg": c}, ck_path)
        torch.save({"model_state_dict": model.state_dict(), "model_config": {**mc, "trained_steps": step, "tokens": tokens}, "step": step}, rdir / "final_model.pt")
    log.close(); tail = step_times[2:] if len(step_times) > 3 else step_times
    info = dict(history=hist, tok_s=float(T / np.median(tail)), params=n_params, steps=step, tokens=tokens, minutes=(time.time() - t0) / 60, model=model if keep_model else None,
                mix={d: v / max(1, sum(stream.dom_tokens.values())) for d, v in stream.dom_tokens.items()})
    del opt, scaler; gc.collect()
    if device.type == "cuda": torch.cuda.empty_cache()
    return info

def plan_budget(arch, tok_s, hours):
    c = {**COMMON, **ARCH_CFG[arch]}; per = c["micro_bs"] * SEQ * c["accum"]; steps = int(hours * 3600 * tok_s / per) // 10 * 10
    print(f"  {arch:11s} at {tok_s/1e3:5.1f}k tok/s, {hours} GPU-h = {steps} steps = {steps*per/1e9:.2f}B tokens"); return steps

def run_arch(arch, hours=None, steps=None, run=None, **over):
    """probe the real throughput, size the schedule to the GPU-hour budget, then train (resumes automatically)"""
    hours = HOURS[arch] if hours is None else hours
    probe = train(arch, cfg=dict(init="scratch"), steps=8, accum=2, do_eval=False, save=False, resume=False, quiet=True, run="probe_" + arch)
    steps = steps or plan_budget(arch, probe["tok_s"], hours)
    cfg = dict(total_steps=steps, warmup=max(20, min(150, steps // 20)), decay_start=int(0.7 * steps), **over)
    print(f"[{arch}] {steps} steps for ~{hours} GPU-hours (measured {probe['tok_s']/1e3:.1f}k tok/s)")
    return train(arch, cfg=cfg, run=run)


In [ ]:
# ================= CELL 6 — SMOKE TEST: every architecture, 6 optimizer steps + validation (~5-8 min total) =================
rows = []
for arch in ARCHS:
    info = train(arch, cfg=dict(init="scratch"), steps=6, accum=2, do_eval=True, save=False, resume=False, max_val_batches=2, run="smoke_" + arch)
    rows.append(dict(arch=arch, params_M=info["params"] / 1e6, tok_per_s=info["tok_s"], val_overall=info["history"][-1]["val"]["overall"], finite=bool(np.isfinite(info["history"][-1]["val"]["overall"])))); gc.collect()
    if device.type == "cuda": torch.cuda.empty_cache()
df = pd.DataFrame(rows); print("\n", df.round(3).to_string(index=False)); assert df["finite"].all(), "an architecture produced a non-finite validation loss - do not train it before fixing (try amp=False in ARCH_CFG)"
print("\nGPU-hour planning (steps for the suggested split):")
for _, r in df.iterrows(): plan_budget(r["arch"], r["tok_per_s"], HOURS[r["arch"]])


In [ ]:
# ================= CELL 7 — RUNS. One architecture per cell; each probes the real GPU speed, sizes its schedule to the GPU-hour budget, and resumes automatically =================
# Gate for EVERY run: within ~20 minutes the printed validation loss must be falling steadily. If it is flat or non-finite, stop and send me the log.
# Budgets live in HOURS (cell 1). Run the encoder first: it is the most useful model for Phase 2/3.


In [ ]:
# --- 7a: ENCODER-ONLY (bidirectional Transformer, masked-language modelling) ---
info_enc = run_arch("modernbert")          # default HOURS['modernbert']; pass hours=0.4 for a 25-minute pilot first


In [ ]:
# --- 7b: DECODER-ONLY (causal Transformer, next-token prediction) ---
info_dec = run_arch("gpt2")                # to continue the old 3,000-step weights instead of starting fresh: run_arch("gpt2", init="warm3000")


In [ ]:
# --- 7c: ENCODER-DECODER (T5-style Transformer, span denoising with the single [MASK] sentinel) ---
info_t5 = run_arch("t5")                   # if the smoke test showed non-finite losses (fp16 overflow is common in T5), set ARCH_CFG['t5']['amp'] = False first


In [ ]:
# --- 7d: RECURRENT (bidirectional LSTM language model, ELMo-style; no attention at all) ---
info_rnn = run_arch("bilstm")


In [ ]:
# ================= CELL 8 — export: files for the Phase 2/3 notebooks + a loader module that handles all four architectures =================
LOADER_SRC = r'''"""arch_loaders.py - load any of the four pre-trained architectures as a backbone for classification / tagging.
   load_backbone(path) -> (backbone, hidden_size, causal, max_len, family)
   backbone(input_ids, attention_mask) -> object with .last_hidden_state of shape [B, T, hidden_size]"""
import torch, torch.nn as nn
from types import SimpleNamespace
from transformers import GPT2Config, GPT2LMHeadModel, ModernBertConfig, ModernBertForMaskedLM, T5Config, T5ForConditionalGeneration

class BiLSTMLM(nn.Module):
    def __init__(self, vocab, emb=512, hidden=1024, layers=3):
        super().__init__()
        self.vocab, self.emb_dim, self.hidden, self.layers = vocab, emb, hidden, layers
        self.emb = nn.Embedding(vocab, emb); self.fwd = nn.LSTM(emb, hidden, layers, batch_first=True); self.bwd = nn.LSTM(emb, hidden, layers, batch_first=True)
        self.norm_f, self.norm_b = nn.LayerNorm(hidden), nn.LayerNorm(hidden)
        self.proj_f, self.proj_b = nn.Linear(hidden, emb), nn.Linear(hidden, emb); self.out_bias = nn.Parameter(torch.zeros(vocab))
    def hidden_states(self, input_ids, attention_mask=None):
        x = input_ids; B, T = x.shape
        lengths = attention_mask.sum(1) if attention_mask is not None else torch.full((B,), T, device=x.device)
        rev = (lengths[:, None] - 1 - torch.arange(T, device=x.device)[None, :]).clamp(min=0)
        hf, _ = self.fwd(self.emb(x)); hb, _ = self.bwd(self.emb(x.gather(1, rev)))
        hb = hb.gather(1, rev[:, :, None].expand(-1, -1, hb.shape[-1]))
        return torch.cat([hf, hb], dim=-1)

class Backbone(nn.Module):
    """one calling convention for every family (matches what the Phase 3 SeqClassifier expects)"""
    def __init__(self, family, inner, embed):
        super().__init__(); self.family, self.inner, self._embed = family, inner, embed
    def get_input_embeddings(self): return self._embed
    def forward(self, input_ids=None, attention_mask=None, inputs_embeds=None, use_cache=False, **kw):
        f = self.family
        if f == "gpt2": h = self.inner(input_ids=input_ids, attention_mask=attention_mask, inputs_embeds=inputs_embeds, use_cache=False).last_hidden_state
        else:
            assert inputs_embeds is None, f"prompt tuning (inputs_embeds) is only wired for the gpt2 family, not {f}"
            if f in ("modernbert", "t5"): h = self.inner(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
            elif f == "bilstm": h = self.inner.hidden_states(input_ids, attention_mask)
            else: raise ValueError(f)
        return SimpleNamespace(last_hidden_state=h)

CAUSAL = {"gpt2": True, "modernbert": False, "t5": False, "bilstm": False}
# modules to adapt with LoRA / LoRMA / IA3 (None = full fine-tuning only). ModernBERT's q,k,v are ONE fused linear, 'Wqkv' (rows: q | k | v).
ADAPTER_TARGETS = {"gpt2": "attn.c_attn (fused Conv1D: q|k|v)", "modernbert": "attn.Wqkv (fused Linear: q|k|v)", "t5": "SelfAttention q, k, v, o (Linear)", "bilstm": None}

def build_full(mc):
    f, V, d = mc["family"], mc["vocab_size"], mc["hidden_size"]
    if f == "gpt2":
        return GPT2LMHeadModel(GPT2Config(vocab_size=V, n_positions=mc["context_length"], n_ctx=mc["context_length"], n_embd=d, n_layer=mc["num_layers"], n_head=mc["num_heads"], n_inner=mc.get("n_inner", 4 * d),
                                          resid_pdrop=0.0, embd_pdrop=0.0, attn_pdrop=0.0))
    if f == "modernbert":
        return ModernBertForMaskedLM(ModernBertConfig(vocab_size=V, max_position_embeddings=mc["context_length"], hidden_size=d, intermediate_size=mc["intermediate_size"], num_hidden_layers=mc["num_layers"],
                                                      num_attention_heads=mc["num_heads"], local_attention=mc["local_attention"], global_attn_every_n_layers=mc["global_attn_every_n_layers"], pad_token_id=0,
                                                      attention_dropout=0.0, mlp_dropout=0.0, embedding_dropout=0.0, classifier_pooling="mean", sparse_prediction=True, sparse_pred_ignore_index=-100, reference_compile=False))
    if f == "t5":
        return T5ForConditionalGeneration(T5Config(vocab_size=V, d_model=d, d_kv=mc["d_kv"], d_ff=mc["d_ff"], num_layers=mc["num_layers"], num_decoder_layers=mc["num_decoder_layers"], num_heads=mc["num_heads"],
                                                   feed_forward_proj="gated-gelu", dropout_rate=0.0, tie_word_embeddings=True))
    if f == "bilstm": return BiLSTMLM(V, emb=mc["emb"], hidden=mc["lstm_hidden"], layers=mc["num_layers"])
    raise ValueError(f)

def load_backbone(path, device="cpu"):
    ck = torch.load(path, map_location="cpu", weights_only=False); mc = ck["model_config"]; full = build_full(mc); full.load_state_dict(ck["model_state_dict"], strict=True)
    f = mc["family"]
    if f == "gpt2": inner, embed = full.transformer, full.transformer.wte
    elif f == "modernbert": inner, embed = full.model, full.model.embeddings.tok_embeddings
    elif f == "t5": inner, embed = full.encoder, full.shared
    elif f == "bilstm": inner, embed = full, full.emb
    else: raise ValueError(f)
    return Backbone(f, inner, embed).to(device), mc["hidden_size"], CAUSAL[f], mc["context_length"], f
'''
(OUT / "arch_loaders.py").write_text(LOADER_SRC); print("wrote", OUT / "arch_loaders.py")
sys.path.insert(0, str(OUT)); import importlib, arch_loaders; importlib.reload(arch_loaders)
print(f"\n{'architecture':12s} {'file':58s} {'steps':>7s} {'params':>8s}")
for arch in ARCHS:
    p = OUT / arch / "final_model.pt"
    if not p.exists(): print(f"{arch:12s} (not trained yet)"); continue
    bb, hidden, causal, ctx, fam = arch_loaders.load_backbone(str(p)); ck = torch.load(p, map_location="cpu", weights_only=False, mmap=True)
    ids = torch.randint(4, 1000, (2, 24)); mask = torch.ones_like(ids); mask[1, 16:] = 0
    with torch.no_grad(): h = bb(input_ids=ids, attention_mask=mask).last_hidden_state
    assert h.shape == (2, 24, hidden) and torch.isfinite(h).all(), arch
    print(f"{arch:12s} {str(p):58s} {ck['step']:>7d} {sum(q.numel() for q in bb.parameters())/1e6:7.1f}M | backbone output {tuple(h.shape)} | causal={causal} | adapters: {arch_loaders.ADAPTER_TARGETS[fam]}")
    print(f'   Phase 3 registry ->  ARCHS["{arch}"] = dict(path="{p}", family="{fam}")')
print("\nUse arch_loaders.load_backbone(path) in the Phase 2/3 notebooks and the tokenizer file legal_bpe_32k_mask.json (vocabulary 32001).")
